# FakeShield — Data Preprocessing
**CSC-233: AI Lab | Beaconhouse National University | Spring 2026**

**Group:** Abdullah Irfan (Lead), Tayyeba Sharafat, Danish Tariq, Syed Mohsin Salman

---

This shared notebook prepares the data for **all four models**. Run this **once first**. It cleans the text, builds the TF-IDF features, and saves the train/test splits so every member trains on the *same data*.

### How to use
1. Download the dataset from [Kaggle](https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset)
2. Upload `Fake.csv` and `True.csv` to this Colab session (or mount Drive)
3. Run all cells top to bottom

## Step 1 — Import libraries

In [ ]:
import pandas as pd
import numpy as np
import re
import pickle

import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

print('Libraries imported successfully!')

## Step 2 — Load the dataset
We label **Fake = 1** and **Real = 0**, then combine and shuffle both files.

In [ ]:
fake_df = pd.read_csv('Fake.csv')
true_df = pd.read_csv('True.csv')

fake_df['label'] = 1   # 1 = Fake
true_df['label'] = 0   # 0 = Real

df = pd.concat([fake_df, true_df], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle

print('Total samples :', len(df))
print('Fake articles :', fake_df.shape[0])
print('Real articles :', true_df.shape[0])
df.head()

## Step 3 — Clean the text
We lowercase everything, remove URLs, punctuation, numbers, and English stopwords. The cleaned `title + text` becomes our model input.

In [ ]:
STOP_WORDS = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)   # remove URLs
    text = re.sub(r'[^a-z\s]', '', text)          # keep letters only
    text = ' '.join(w for w in text.split() if w not in STOP_WORDS)
    return text

df['content'] = df['title'] + ' ' + df['text']
print('Cleaning text... (this may take 1-2 minutes)')
df['clean_content'] = df['content'].apply(clean_text)
print('Done!')
df[['clean_content', 'label']].head()

## Step 4 — Save the processed data
This `processed_data.csv` is one of the files we upload to GitHub.

In [ ]:
df[['clean_content', 'label']].to_csv('processed_data.csv', index=False)
print('Saved: processed_data.csv')

## Step 5 — TF-IDF Vectorization + Train/Test Split
We convert text into numeric features using TF-IDF (top 50,000 features, unigrams + bigrams), then split 80% train / 20% test. The **same** vectorizer and splits are reused by every model.

In [ ]:
X = df['clean_content']
y = df['label']

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1, 2))
X_train = tfidf.fit_transform(X_train_raw)
X_test  = tfidf.transform(X_test_raw)

print('Training set shape:', X_train.shape)
print('Testing set shape :', X_test.shape)
print('Vocabulary size   :', len(tfidf.vocabulary_))

## Step 6 — Save everything for the model notebooks
Each member's notebook will load these files so all models train on identical data.

In [ ]:
pickle.dump(tfidf,   open('tfidf_vectorizer.pkl', 'wb'))
pickle.dump(X_train, open('X_train.pkl', 'wb'))
pickle.dump(X_test,  open('X_test.pkl',  'wb'))
pickle.dump(y_train, open('y_train.pkl', 'wb'))
pickle.dump(y_test,  open('y_test.pkl',  'wb'))

print('All files saved successfully:')
print('  tfidf_vectorizer.pkl  <- needed by the backend')
print('  X_train.pkl / X_test.pkl')
print('  y_train.pkl / y_test.pkl')
print('  processed_data.csv    <- upload to GitHub')

print('\nNext: each member runs their own model notebook.')